# Phase 4: Gen AI explanation layer + faithfulness audit

It is common to check "why is this patient flagged" next to
a risk score; but uncommon to check whether it's faithful to the model. Here, I check:

LightGBM predicts → SHAP gives per patient signed drivers → an LLM writes
a clinician-readable explanation → a separate LLM pass parses it back into
claims → audit those claims against the SHAP ground truth.

Four countable checks: feature grounding, direction fidelity (the headline),
unsupported clinical claims, and stability.

**Requires** an OpenAI key. Install extras and set the key first:
```bash
make setup-genai
cp .env.example .env   # then add OPENAI_API_KEY
```

In [1]:
%run data_prep.ipynb   # constants + read/cohort/feature/split functions

import os
import numpy as np
import lightgbm as lgb
import shap
from dotenv import load_dotenv
from openai import OpenAI
import instructor
from pydantic import BaseModel, Field
from typing import Literal

load_dotenv(PROJECT_ROOT / '.env')
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in .env first.'
MODEL = 'gpt-4o-mini'
client = OpenAI()

data_prep loaded | 101,766 rows, 71518 patients, 1.42 encounters/patient
   encounter_id  patient_nbr             race  gender      age weight  \
0       2278392      8222157        Caucasian  Female   [0-10)    NaN   
1        149190     55629189        Caucasian  Female  [10-20)    NaN   
2         64410     86047875  AfricanAmerican  Female  [20-30)    NaN   
3        500364     82442376        Caucasian    Male  [30-40)    NaN   
4         16680     42519267        Caucasian    Male  [40-50)    NaN   

   admission_type_id  discharge_disposition_id  admission_source_id  \
0                  6                        25                    1   
1                  1                         1                    7   
2                  1                         1                    7   
3                  1                         1                    7   
4                  1                         1                    7   

   time_in_hospital  ... citoglipton insulin  glyburide-metfo

/Users/ramibaghdan/Desktop/readmission-eval/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Train the corrected model and compute SHAP drivers

Same corrected setup as the main notebook (grouped split + cohort exclusions).

In [2]:
raw = load_raw()
df, _ = build_cohort(raw, exclude_expired_hospice=True)
X, cats, meta = build_features(df)
y, groups = df[TARGET], df[PATIENT_ID]
tr, te = grouped_split(y, groups)

model = lgb.LGBMClassifier(objective='binary', n_estimators=600, learning_rate=0.03,
    num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
    min_child_samples=50, random_state=SEED, n_jobs=-1, verbose=-1)
model.fit(X.iloc[tr], y.iloc[tr], categorical_feature=cats)

X_test = X.iloc[te].reset_index(drop=True)
y_prob = model.predict_proba(X_test)[:, 1]

shap_values = shap.TreeExplainer(model).shap_values(X_test)
if isinstance(shap_values, list):
    shap_values = shap_values[1]
shap_values = np.asarray(shap_values)
print('SHAP matrix:', shap_values.shape)

SHAP matrix: (19802, 44)


/Users/ramibaghdan/Desktop/readmission-eval/.venv/lib/python3.13/site-packages/shap/explainers/_tree.py:448: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn('LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray')


## Top 8 most influential features per patient and pick the highest risk patient to feed into LLM

In [3]:

TOP_K = 8

def top_drivers(row):
    'Signed top-k SHAP drivers for one test row (the ground truth).'
    contribs = shap_values[row]
    order = np.argsort(-np.abs(contribs))[:TOP_K]
    return [{'feature': X_test.columns[j], 'value': X_test.iloc[row, j],
             'shap': float(contribs[j]),
             'direction': 'increases' if contribs[j] > 0 else 'decreases'} for j in order]

example = top_drivers(int(np.argsort(-y_prob)[0]))  # highest-risk patient
example

[{'feature': 'number_inpatient',
  'value': 12,
  'shap': 2.462966182094722,
  'direction': 'increases'},
 {'feature': 'age',
  'value': '[20-30)',
  'shap': 0.26891844563656814,
  'direction': 'increases'},
 {'feature': 'payer_code',
  'value': 'OG',
  'shap': 0.260575909418877,
  'direction': 'increases'},
 {'feature': 'number_emergency',
  'value': 3,
  'shap': 0.17262304844419657,
  'direction': 'increases'},
 {'feature': 'num_lab_procedures',
  'value': 11,
  'shap': 0.12996521032589553,
  'direction': 'increases'},
 {'feature': 'discharge_disposition_id',
  'value': 1,
  'shap': -0.07903888561421808,
  'direction': 'decreases'},
 {'feature': 'insulin',
  'value': 'Up',
  'shap': 0.052166280683466904,
  'direction': 'increases'},
 {'feature': 'diag_1_group',
  'value': 'Diabetes',
  'shap': 0.049677871701643506,
  'direction': 'increases'}]

## Generate an explanation, then parse it back into structured claims

Two separate LLM calls: one writes prose, the other
extracts (feature, direction) claims from that prose.

In [4]:
#Feature glossary: plain-English meaning for each model feature ---
# Used by BOTH the generator (so it names factors correctly and never invents)
# and the extractor (so legitimate rephrasings map to canonical names, not INVENTED).
FEATURE_GLOSSARY = {
    'time_in_hospital': 'length of the hospital stay, in days',
    'num_lab_procedures': 'number of lab tests done during the stay',
    'num_procedures': 'number of non-lab procedures done during the stay',
    'num_medications': 'number of distinct medications given during the stay',
    'number_outpatient': 'outpatient visits in the year before this stay',
    'number_emergency': 'emergency visits in the year before this stay',
    'number_inpatient': 'inpatient (admitted) stays in the year before this stay',
    'number_diagnoses': 'number of diagnoses recorded for this stay',
    'admission_type_id': 'type of admission (e.g., emergency, elective)',
    'discharge_disposition_id': 'where/how the patient was discharged (e.g., home, another facility, SNF)',
    'admission_source_id': 'source of the admission (e.g., physician referral, ER)',
    'race': 'patient race', 'gender': 'patient gender', 'age': 'patient age band',
    'payer_code': 'insurance / payer type',
    'medical_specialty': 'specialty of the admitting physician',
    'max_glu_serum': 'maximum glucose serum test result',
    'A1Cresult': 'HbA1c (long-term blood sugar) test result',
    'change': 'whether any diabetes medication was changed during the stay',
    'diabetesMed': 'whether any diabetes medication was prescribed',
    'diag_1_group': 'primary diagnosis category',
    'diag_2_group': 'secondary diagnosis category',
    'diag_3_group': 'additional diagnosis category',
}

def describe(feat):
    'Plain-English meaning for a feature name (falls back gracefully).'
    if feat in FEATURE_GLOSSARY:
        return FEATURE_GLOSSARY[feat]
    if feat in MEDICATION_COLUMNS:
        return f'{feat} dosage status during the stay (No / Steady / Up / Down)'
    return feat.replace('_', ' ')


#Tighter generator prompt: only the listed factors, exact direction, no advice ---
def generate_explanation(drivers, risk):
    lines = [f'Predicted 30-day readmission risk: {risk:.1%}',
             'Risk drivers (the ONLY factors you may discuss):']
    lines += [f"  - {d['feature']} ({describe(d['feature'])}) = {d['value']!r}: {d['direction']} risk"
              for d in drivers]
    system = (
        'You are a clinical decision-support assistant. In 2-3 sentences, explain to a clinician '
        'why the model produced this risk score.\n'
        'STRICT RULES:\n'
        '1. Discuss ONLY the risk drivers listed. Do NOT mention any diagnosis, medication, lab, '
        'comorbidity, or clinical concept that is not in the list.\n'
        '2. Refer to each factor by its canonical name (the identifier before the parentheses); '
        'you may use the description for readability, but introduce no new factors.\n'
        "3. State each factor's direction (raises vs lowers risk) EXACTLY as given; never flip or infer it.\n"
        '4. Give NO recommendations, prognoses, or treatment advice.')
    resp = client.chat.completions.create(model=MODEL, temperature=0.0, messages=[
        {'role': 'system', 'content': system},
        {'role': 'user', 'content': '\n'.join(lines) + '\n\nWrite the explanation now.'}])
    return resp.choices[0].message.content.strip()

class FactorClaim(BaseModel):
    feature: str = Field(description="Canonical feature name, or 'INVENTED' if it maps to none provided.")
    stated_direction: Literal['increases', 'decreases', 'unclear']

class ExtractedClaims(BaseModel):
    referenced_factors: list[FactorClaim] = Field(default_factory=list)
    unsupported_clinical_claims: list[str] = Field(default_factory=list)

ins = instructor.from_openai(client)

#Extractor gets the glossary so rephrasings map to canonical names (not INVENTED)
def extract_claims(narrative, feature_names):
    glossary = '\n'.join(f'- {f}: {describe(f)}' for f in sorted(feature_names))
    return ins.chat.completions.create(model=MODEL, temperature=0.0, response_model=ExtractedClaims,
        messages=[
            {'role': 'system', 'content':
             'Extract structured claims from a clinical explanation. You are given a glossary of '
             'canonical model features with plain-English descriptions. For every risk factor the '
             'explanation discusses, map it to the canonical feature whose name OR description it '
             'refers to (e.g. "frequent prior hospitalizations" -> number_inpatient; "discharged '
             'home" -> discharge_disposition_id). Set feature="INVENTED" ONLY if it genuinely '
             'matches no glossary entry. Also list any prognostic/treatment assertions not '
             'derivable from these features.'},
            {'role': 'user', 'content': f'Canonical feature glossary:\n{glossary}\n\n'
             f'Explanation:\n\"\"\"\n{narrative}\n\"\"\"'}])

narrative = generate_explanation(example, y_prob[int(np.argsort(-y_prob)[0])])
print(narrative)
extract_claims(narrative, list(X_test.columns))

The predicted 30-day readmission risk of 77.8% is influenced by several key risk drivers. The number of inpatient stays in the year before this stay (12), the age band of 20-30 years, the payer code 'OG', the number of emergency visits (3), the number of lab procedures (11), and the insulin dosage status being 'Up' all increase the risk. Conversely, the discharge disposition being '1' decreases the risk, contributing to the overall assessment.


ExtractedClaims(referenced_factors=[FactorClaim(feature='number_inpatient', stated_direction='increases'), FactorClaim(feature='age', stated_direction='increases'), FactorClaim(feature='payer_code', stated_direction='increases'), FactorClaim(feature='number_emergency', stated_direction='increases'), FactorClaim(feature='num_lab_procedures', stated_direction='increases'), FactorClaim(feature='insulin', stated_direction='increases'), FactorClaim(feature='discharge_disposition_id', stated_direction='decreases')], unsupported_clinical_claims=['The predicted 30-day readmission risk of 77.8% is influenced by several key risk drivers.'])

## Audit a batch and read the headline numbers

For each patient: generate twice (stability at temp 0), extract claims, and
check each claim's direction against the SHAP sign.

In [5]:
from tqdm import tqdm

rng = np.random.default_rng(SEED)
N = 50
top_pool = np.argsort(-y_prob)[:N * 4]
rows = sorted(set(rng.choice(top_pool, N // 2, replace=False).tolist())
              | set(rng.choice(len(y_prob), N - N // 2, replace=False).tolist()))[:N]

n_rev = n_checked = n_inv = n_ref = flipped_expl = 0
for row in tqdm(rows):
    drivers = top_drivers(row)
    truth = {d['feature']: d['direction'] for d in drivers}
    narr = generate_explanation(drivers, y_prob[row])
    claims = extract_claims(narr, list(X_test.columns))
    row_flip = False
    for fc in claims.referenced_factors:
        n_ref += 1
        if fc.feature not in truth:
            n_inv += 1
            continue
        if fc.stated_direction in ('increases', 'decreases'):
            n_checked += 1
            if fc.stated_direction != truth[fc.feature]:
                n_rev += 1; row_flip = True
    flipped_expl += int(row_flip)

print(f'\nexplanations audited            : {len(rows)}')
print(f'% reversing a risk direction    : {100 * flipped_expl / len(rows):.1f}%')
print(f'direction-reversal rate (claims): {n_rev / n_checked:.3f}' if n_checked else 'n/a')
print(f'invented-feature rate           : {n_inv / n_ref:.3f}' if n_ref else 'n/a')

100%|███████████████████████████████████████████| 50/50 [03:52<00:00,  4.65s/it]


explanations audited            : 50
% reversing a risk direction    : 2.0%
direction-reversal rate (claims): 0.003
invented-feature rate           : 0.028
